# 02 SpaceX Falcon 9 - Data Collection (Wikipedia Web Scraping)

This notebook is Step 02 in a multi-step end-to-end data science project that analyzes and predicts first-stage landing success for SpaceX Falcon 9 launches.

**Pipeline overview:**

- **Step 01:** Collect launch data from the SpaceX REST API and create an initial modeling dataset.

- **Step 02 (this notebook):** Scrape a *fixed Wikipedia revision* of Falcon 9 & Falcon Heavy launch tables and export a clean CSV for downstream wrangling/EDA/modeling.

- **Next steps:** Data wrangling, exploratory analysis (SQL + visualization), interactive geospatial map, dashboarding, and machine learning modeling.

**Primary output:** `../data/processed/02_wikipedia_launches.csv`

**Raw data snapshot (for reproducibility):** `../data/raw/02_wikipedia_falcon9_launches_1027686922.thml` (Wikipedia `oldid = 1027686922`).

# Notebook sections

**1. Setup**

**2. Fetch the Wikipedia HTML**

**3. Parse and extract launch records**

**4. Save artifacts**

**5. Assumptions**

---

## 1. Setup

In [1]:
from pathlib import Path

import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# Paths

PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'

RAW_DIR = DATA_DIR / 'raw'
RAW_DIR.mkdir(parents = True, exist_ok = True)

PROCESSED_DIR = DATA_DIR / 'processed'
PROCESSED_DIR.mkdir(parents = True, exist_ok = True)

# Output artifact paths
WIKI_OLDID = '1027686922'
RAW_HTML_PATH = RAW_DIR / f'02_wikipedia_falcon9_launches_{WIKI_OLDID}.html'
PROCESSED_CSV_PATH = PROCESSED_DIR / '02_wikipedia_launches.csv'

### 1.1 Helper functions

Wikipedia tables contain footnotes, links, and formatting quirks. The helper functions below standardize how we extract:

- date/time fields

- booster version

- payload mass

- column names from `<th>` headers

In [2]:
def date_time(table_cells):
    """
    This function returns the data and time from the HTML table cell
    Input: the  element of a table data cell extracts extra row
    """
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """
    This function returns the booster version from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out = ''.join([booster_version for i, booster_version in enumerate(table_cells.strings) if i % 2 == 0][0: -1])
    return out

def landing_status(table_cells):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out = [i for i in table_cells.strings][0]
    return out


def get_mass(table_cells):
    mass = unicodedata.normalize('NFKD', table_cells.text).strip()
    if mass:
        mass.find('kg')
        new_mass = mass[0:mass.find('kg')+2]
    else:
        new_mass = 0
    return new_mass


def extract_column_from_header(row):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    if (row.br):
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
        
    colunm_name = ' '.join(row.contents)
    
    # Filter the digit and empty names
    if not(colunm_name.strip().isdigit()):
        colunm_name = colunm_name.strip()
        return colunm_name

We scrape a fixed Wikipedia revision to keep this dataset stable across runs.

- Page snapshot date: **June 9, 2021**

- `oldid`: **1027686922**

In [3]:
static_url = (
    'https://en.wikipedia.org/w/index.php'
    '?title=List_of_Falcon_9_and_Falcon_Heavy_launches'
    f'&oldid={WIKI_OLDID}'
)

headers = {
    'User-Agent': (
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/91.0.4472.124 Safari/537.36'
    )
}

## 2. Fetch the Wikipedia HTML

We fetch the fixed-revision page and store the HTML to `../data/raw` so the notebook can be re-run without relying on the live web.

### 2.1 Request the page

In [4]:
# Fetch the page once and cache the raw HTML for reproducibility.
# If the cached file exists, we reuse it to make the notebook re-runnable offline.
if RAW_HTML_PATH.exists():
    html_text = RAW_HTML_PATH.read_text(encoding = 'utf-8')
    source = f'cache: {RAW_HTML_PATH}'
else:
    response = requests.get(static_url, headers = headers, timeout = 30)
    response.raise_for_status()
    html_text = response.text
    RAW_HTML_PATH.write_text(html_text, encoding = 'utf-8')
    source = f'download: {static_url}'

## 3. Parse the page with BeautifulSoup

We parse the cached HTML with BeautifulSoup, identify the launch record tables, extract column names, and then iterate through the table rows to build a structured dataset.

In [5]:
# Create a BeautifulSoup object from the HTML content
soup = BeautifulSoup(html_text, 'html.parser')

# Verify the page title looks correct
soup.title

<title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>

### 3.1 Identify the launch record table and extract column names

Wikipedia pages often contain many tables; we first locate the launch record tables. Then extract the column names from the table header.

In [6]:
# Find all tables on the page
html_tables = soup.find_all('table')
len(html_tables)

25

In [7]:
# The launch record tables start from the 3rd table on this page snapshot
first_launch_table = html_tables[2]

# Preview a few header cells (instead of printing the full HTML)
preview_headers = [extract_column_from_header(th) for th in first_launch_table.find_all('th')[:12]]
preview_headers

['Flight No.',
 'Date and time ( )',
 '',
 'Launch site',
 'Payload',
 'Payload mass',
 'Orbit',
 'Customer',
 'Launch outcome',
 '',
 None,
 None]

We will not print the full HTML table. Instead, we preview a few header labels. Extract header labels (column names) into a Python list.

In [8]:
column_names = []

# Apply find_all() function with `th` element on first_launch_table
th_elements = first_launch_table.find_all('th')

# Iterate through th_elements to extract each column name
for th in th_elements:
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)

Preview a subset of extracted column names.

In [9]:
# Display the first 15 column names and total count
column_names[:15], len(column_names)

(['Flight No.',
  'Date and time ( )',
  'Launch site',
  'Payload',
  'Payload mass',
  'Orbit',
  'Customer',
  'Launch outcome'],
 8)

### 3.2 Parse the launch record tables into a dataset

Build a dictionary of columns - list of values, then convert it into a Pandas DataFrame.

In [10]:
launch_dict= dict.fromkeys(column_names)

# Remove 'Date and time' key (we store Date and Time separately below).
for k in list(launch_dict.keys()):
    if k.startswith('Date and time'):
        del launch_dict[k]
        break

# Initialize the columns as empty lists
launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []

# Added columns (parsed from the table)
launch_dict['Version Booster'] = []
launch_dict['Booster landing'] = []
launch_dict['Date'] = []
launch_dict['Time'] = []

Next, we iterate through each launch table row and extract the relevant fields.

In [11]:
extracted_row = 0
# Iterate through all tables with class `wikitable plainrowheaders collapsible`
for table_number, table in enumerate(soup.find_all('table', 'wikitable plainrowheaders collapsible')):
    # Iterate through all rows in the table
    for rows in table.find_all('tr'):
        # Check to see if first table heading is as number corresponding to launch a number 
        if rows.th:
            if rows.th.string:
                flight_number = rows.th.string.strip()
                flag = flight_number.isdigit()
        else:
            flag = False
        # Get all `td` elements for the current row
        row = rows.find_all('td')
        # If it is number save cells in a dictionary
        if flag:
            extracted_row += 1
            # Flight Number value
            launch_dict['Flight No.'].append(flight_number)

            datatimelist = date_time(row[0])

            # Date value
            date = datatimelist[0].strip(',')
            launch_dict['Date'].append(date)
            
            # Time value
            time = datatimelist[1]
            launch_dict['Time'].append(time)
              
            # Booster version
            bv = booster_version(row[1])
            if not(bv):
                bv = row[1].a.string
            launch_dict['Version Booster'].append(bv)
            
            # Launch Site
            launch_site = row[2].a.string
            launch_dict['Launch site'].append(launch_site)
            
            # Payload
            payload = row[3].a.string
            launch_dict['Payload'].append(payload)
            
            # Payload Mass
            payload_mass = get_mass(row[4])
            launch_dict['Payload mass'].append(payload_mass)
            
            # Orbit
            orbit = row[5].a.string
            launch_dict['Orbit'].append(orbit)
            
            # Customer
            if row[6].a:
                customer = row[6].a.string
            else:
                customer = row[6].text.strip()
            launch_dict['Customer'].append(customer)
            
            # Launch outcome
            launch_outcome = list(row[7].strings)[0]
            launch_dict['Launch outcome'].append(launch_outcome)
            
            # Booster landing
            booster_landing = landing_status(row[8])
            launch_dict['Booster landing'].append(booster_landing)

# Display number of launches extracted
extracted_row

121

Convert the parsed dictionary into a DataFrame.

In [12]:
df = pd.DataFrame({key: pd.Series(value) for key, value in launch_dict.items()})

# Clean up stray newlines/whitespace from wiki cells
TEXT_COLS = [
    'Launch outcome',
    'Booster landing',
    'Launch site',
    'Payload',
    'Orbit',
    'Customer',
    'Version Booster'
]

for col in TEXT_COLS:
    if col in df.columns:
        df[col] = df[col].apply(
            lambda x: re.sub(r'\s+', ' ', x.replace('\xa0', ' ').replace('\n', ' ')).strip()
            if isinstance(x, str) else x
        )

df.head()

,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
0,1,CCAFS,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,F9 v1.07B0003.18,Failure,4 June 2010,18:45
1,2,CCAFS,Dragon,0,LEO,NASA,Success,F9 v1.07B0004.18,Failure,8 December 2010,15:43
2,3,CCAFS,Dragon,525 kg,LEO,NASA,Success,F9 v1.07B0005.18,No attempt,22 May 2012,07:44
3,4,CCAFS,SpaceX CRS-1,"4,700 kg",LEO,NASA,Success,F9 v1.07B0006.18,No attempt,8 October 2012,00:35
4,5,CCAFS,SpaceX CRS-2,"4,877 kg",LEO,NASA,Success,F9 v1.07B0007.18,No attempt,1 March 2013,15:10


## 4. Save artifacts

We save the parsed dataset to `../data/processed` for downstream wrangling and analysis.

In [13]:
df.to_csv(PROCESSED_CSV_PATH, index = False)

## 5. Assumptions

- **Fixed Wikipedia revision:** We scrape a pinned revision (`oldid = 1027686922`) so results are reproducible.

- **Offline re-runs:** If the raw HTML snapshot exists in `../data/raw`, we reuse it instead of hitting Wikipedia.

- **Table parsing:** We rely on the page structure of the pinned revision (launch record tables with the `wikitable plainrowheaders collapsible` class).

- **Payload mass parsing:** Payload mass is kept as a string (e.g., `'4,700 kg'`) or `0` if missing in the source table. Numeric conversion happens later during wrangling.

- **Date/time fields:** Data and time are stored as strings exactly as shown in the Wikipedia table; downstream notebooks standardize to a machine-friendly format.

---